# Шаг 02 — чат: история и стриминг

Модель ничего не помнит между вызовами. Проверим:

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
print(llm.invoke("Меня зовут Женя.").content)
print("---")
print(llm.invoke("Как меня зовут?").content)

«Память» чата — это **список сообщений**, который мы отправляем целиком на каждом ходу:

- `SystemMessage` — правила;
- `HumanMessage` — реплика пользователя;
- `AIMessage` — ответ модели (тоже кладём в историю).

А `.stream()` отдаёт ответ кусками по мере генерации.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

history = [SystemMessage("Ты полезный ассистент. Отвечай кратко.")]


def chat(text: str) -> None:
    history.append(HumanMessage(text))
    reply = ""
    for chunk in llm.stream(history):          # chunks arrive as they are generated
        print(chunk.content, end="", flush=True)
        reply += chunk.content
    history.append(AIMessage(reply))          # remember the answer

In [ ]:
chat("Меня зовут Женя.")

In [ ]:
chat("Как меня зовут?")

Вот что на самом деле уходит в модель на каждом ходу:

In [ ]:
for message in history:
    print(f"{message.type:>6}: {message.content}")